In [1]:
# Import the necessary libraries
#--------------------------------
import pandas as pd
import geopandas as gpd
import numpy as np
import math

In [2]:
# Load input data (ie ward boundaries and building footprints)
#--------------------------------------------------------------
gdf_boundaries = gpd.read_file(r"Churu_Ward_Boundaries_MHT.geojson")
buildings_gdf = gpd.read_parquet(r"Churu_buildings_with_OSM.parquet")

In [3]:
buildings_gdf

,perimeter_in_meters,building_faces,bf_source,confidence_left,geometry,longitude,latitude,id,area_in_meters,height_mean,...,First_AID_FA_vehicle,FA_vehicle_distance_km,FA_vehicle_time_min,District_Hospital_DH_pedestrian,DH_pedestrian_distance_km,DH_pedestrian_time_min,First_AID_FA_pedestrian,FA_pedestrian_distance_km,FA_pedestrian_time_min,imputed_from
0,7.243843,4,google,0.7724,"POLYGON ((74.23236 27.72641, 74.23236 27.72641...",74.232350,27.726409,74.23234975129046:27.72640873664523,0.000125,0.000000,...,Luhara - phc,0.07,0.2,District hospital Nagaur - dis_h,88.69,1064.3,Luhara - phc,0.07,0.8,None
1,9.011801,4,google,0.7108,"POLYGON ((74.84492 28.873, 74.84492 28.87303, ...",74.844915,28.873013,74.84491545220574:28.873013019801967,0.000126,1.000000,...,Sahawa - chc,0.19,0.5,D B Government Hospital Churu - dis_h,87.27,1047.2,Sahawa - chc,0.19,2.3,None
2,6.354292,4,google,0.7123,"POLYGON ((74.22657 27.63066, 74.22656 27.63067...",74.226555,27.630664,74.22655544425939:27.630663527650903,0.000126,1.722222,...,Khara - sub_cen,4.21,10.1,District hospital Nagaur - dis_h,80.56,966.7,Parawa - sub_cen,2.90,34.8,None
3,6.957619,4,google,0.7604,"POLYGON ((75.01403 28.33486, 75.01403 28.33487...",75.014020,28.334861,75.0140202674914:28.334861398658834,0.000126,0.000000,...,Boontia - sub_cen,1.85,4.6,D B Government Hospital Churu - dis_h,8.53,102.4,Boontia - sub_cen,1.85,22.2,None
4,8.217702,4,google,0.7845,"POLYGON ((74.5701 27.78267, 74.5701 27.7827, 7...",74.570094,27.782683,74.57009354210085:27.782682537523247,0.000127,2.500000,...,Badawar - phc,0.31,0.5,"S K Hospital, Sikar - dis_h",69.99,839.8,Badawar - phc,0.31,3.8,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1481129,288.063775,4,google,0.9623,"POLYGON ((74.47232 28.41938, 74.4719 28.42, 74...",74.471835,28.419541,74.47183471063026:28.41954127037166,0.255824,10.031507,...,Sardarsahar - chc,3.43,5.0,D B Government Hospital Churu - dis_h,66.48,797.7,Sardarsahar - chc,3.43,41.2,None
1481130,309.362995,6,google,0.8996,"POLYGON ((74.96017 28.19513, 74.9602 28.19508,...",74.960137,28.195540,74.96013678708111:28.195540360926653,0.275961,6.671840,...,Ratannagar - chc,2.31,2.5,D B Government Hospital Churu - dis_h,12.93,155.2,Ratannagar - chc,1.83,22.0,None
1481131,444.405058,21,google,0.9111,"POLYGON ((75.37975 28.63027, 75.37973 28.6301,...",75.380226,28.630323,75.38022567723831:28.630323167273463,0.276669,8.208935,...,Rajgarh - chc,2.52,2.5,D B Government Hospital Churu - dis_h,67.89,814.7,Rajgarh - chc,2.17,26.1,None
1481132,382.805856,16,google,0.9185,"POLYGON ((75.3449 28.39026, 75.34491 28.39022,...",75.344910,28.390703,75.34491042205822:28.390703116146657,0.300631,7.102821,...,Rawatsar Kujala - sub_cen,1.71,4.1,B.D.K. Hospital Jhunjhunun - dis_h,33.53,402.4,Rawatsar Kujala - sub_cen,1.71,20.6,None


In [4]:
gdf_boundaries

,OBJECTID,Name,bvnvb,Shape_Leng,Shape_Area,Ward_sqkm,Percentage,Per_inter,geometry
0,1.0,34,34.0,2994.929840,2.419409e+05,0.242,23.967,NaN,"MULTIPOLYGON (((497413.347 3129795.635, 497432..."
1,2.0,1,1.0,8789.786444,2.305656e+06,2.307,4.725,8.626,"MULTIPOLYGON (((492617.58 3130342.225, 492616...."
2,3.0,4,4.0,7979.093807,1.740005e+06,1.741,9.765,3.044,"MULTIPOLYGON (((494540.406 3129533.656, 494538..."
3,4.0,2,2.0,3002.349186,2.590953e+05,0.259,NaN,NaN,"MULTIPOLYGON (((495502.887 3131172.422, 495503..."
4,5.0,5,5.0,1356.361713,7.908319e+04,0.079,NaN,NaN,"MULTIPOLYGON (((495745.314 3130828.325, 495752..."
5,6.0,3,3.0,2044.537291,1.642186e+05,0.164,NaN,NaN,"MULTIPOLYGON (((495890.081 3130893.476, 495867..."
6,7.0,8,8.0,1003.513210,4.125380e+04,0.041,NaN,48.780,"MULTIPOLYGON (((496040.554 3130563.262, 496017..."
7,8.0,6,6.0,1473.824366,9.274238e+04,0.093,NaN,NaN,"MULTIPOLYGON (((496163.12 3130732.687, 496165...."
8,9.0,58,58.0,1678.334455,1.226817e+05,0.123,NaN,NaN,"MULTIPOLYGON (((496270.673 3131181.759, 496243..."
9,10.0,60,60.0,3696.772565,2.942562e+05,0.294,NaN,NaN,"MULTIPOLYGON (((496025.865 3131334.261, 496022..."


In [5]:
# Filter residential buildings 
#------------------------------
churu_res_buildings = buildings_gdf[buildings_gdf["prediction"] == "Residential"]
churu_other_buildings = buildings_gdf[buildings_gdf["prediction"] != "Residential"]
churu_res_buildings["geometry2"] = buildings_gdf["geometry"]

/usr/local/lib/python3.12/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [6]:
# Recreate geometrires from lat/lon (convert to point geometries)
#----------------------------------------------------------------- 
churu_res_buildings = gpd.GeoDataFrame(
    churu_res_buildings,
    geometry=gpd.points_from_xy(churu_res_buildings["longitude"], churu_res_buildings["latitude"]),
    crs="EPSG:4326"
)
churu_res_buildings

,perimeter_in_meters,building_faces,bf_source,confidence_left,geometry,longitude,latitude,id,area_in_meters,height_mean,...,FA_vehicle_distance_km,FA_vehicle_time_min,District_Hospital_DH_pedestrian,DH_pedestrian_distance_km,DH_pedestrian_time_min,First_AID_FA_pedestrian,FA_pedestrian_distance_km,FA_pedestrian_time_min,imputed_from,geometry2
0,7.243843,4,google,0.7724,POINT (74.23235 27.72641),74.232350,27.726409,74.23234975129046:27.72640873664523,0.000125,0.000000,...,0.07,0.2,District hospital Nagaur - dis_h,88.69,1064.3,Luhara - phc,0.07,0.8,None,"POLYGON ((74.23236 27.72641, 74.23236 27.72641..."
1,9.011801,4,google,0.7108,POINT (74.84492 28.87301),74.844915,28.873013,74.84491545220574:28.873013019801967,0.000126,1.000000,...,0.19,0.5,D B Government Hospital Churu - dis_h,87.27,1047.2,Sahawa - chc,0.19,2.3,None,"POLYGON ((74.84492 28.873, 74.84492 28.87303, ..."
2,6.354292,4,google,0.7123,POINT (74.22656 27.63066),74.226555,27.630664,74.22655544425939:27.630663527650903,0.000126,1.722222,...,4.21,10.1,District hospital Nagaur - dis_h,80.56,966.7,Parawa - sub_cen,2.90,34.8,None,"POLYGON ((74.22657 27.63066, 74.22656 27.63067..."
3,6.957619,4,google,0.7604,POINT (75.01402 28.33486),75.014020,28.334861,75.0140202674914:28.334861398658834,0.000126,0.000000,...,1.85,4.6,D B Government Hospital Churu - dis_h,8.53,102.4,Boontia - sub_cen,1.85,22.2,None,"POLYGON ((75.01403 28.33486, 75.01403 28.33487..."
4,8.217702,4,google,0.7845,POINT (74.57009 27.78268),74.570094,27.782683,74.57009354210085:27.782682537523247,0.000127,2.500000,...,0.31,0.5,"S K Hospital, Sikar - dis_h",69.99,839.8,Badawar - phc,0.31,3.8,None,"POLYGON ((74.5701 27.78267, 74.5701 27.7827, 7..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1481070,288.940417,4,google,0.9654,POINT (75.49377 28.83493),75.493774,28.834925,75.49377437705633:28.83492532996736,0.124482,5.496721,...,0.51,0.8,Maharaja Aggarsain Medical College AGROHA - dis_h,53.42,641.0,Nyangal Chhoti - sub_cen,0.51,6.2,None,"POLYGON ((75.49442 28.83487, 75.4944 28.83505,..."
1481075,413.929305,16,microsoft,NaN,POINT (75.48835 28.57079),75.488354,28.570791,75.48835436178372:28.570790796491007,0.128104,0.000000,...,0.87,1.3,B.D.K. Hospital Jhunjhunun - dis_h,74.42,893.1,Kalari - phc,0.87,10.4,None,"POLYGON ((75.48861 28.5707, 75.48865 28.57068,..."
1481083,289.408869,24,google,0.9092,POINT (74.9331 28.20512),74.933096,28.205125,74.93309554461749:28.205124977994227,0.135970,8.185522,...,2.39,3.8,D B Government Hospital Churu - dis_h,13.48,161.8,Ratannagar - chc,2.39,28.6,None,"POLYGON ((74.93284 28.20498, 74.93285 28.20498..."
1481106,351.817061,31,microsoft,NaN,POINT (75.49022 28.56949),75.490217,28.569488,75.49021717721772:28.56948776087451,0.158354,0.000000,...,0.70,1.1,B.D.K. Hospital Jhunjhunun - dis_h,74.26,891.1,Kalari - phc,0.70,8.4,None,"POLYGON ((75.48981 28.56982, 75.48979 28.56983..."


In [7]:
# Match CRS
#------------
churu_res_buildings = churu_res_buildings.to_crs(gdf_boundaries.crs)
churu_res_buildings

,perimeter_in_meters,building_faces,bf_source,confidence_left,geometry,longitude,latitude,id,area_in_meters,height_mean,...,FA_vehicle_distance_km,FA_vehicle_time_min,District_Hospital_DH_pedestrian,DH_pedestrian_distance_km,DH_pedestrian_time_min,First_AID_FA_pedestrian,FA_pedestrian_distance_km,FA_pedestrian_time_min,imputed_from,geometry2
0,7.243843,4,google,0.7724,POINT (424331.627 3067131.713),74.232350,27.726409,74.23234975129046:27.72640873664523,0.000125,0.000000,...,0.07,0.2,District hospital Nagaur - dis_h,88.69,1064.3,Luhara - phc,0.07,0.8,None,"POLYGON ((74.23236 27.72641, 74.23236 27.72641..."
1,9.011801,4,google,0.7108,POINT (484876.34 3193926.563),74.844915,28.873013,74.84491545220574:28.873013019801967,0.000126,1.000000,...,0.19,0.5,D B Government Hospital Churu - dis_h,87.27,1047.2,Sahawa - chc,0.19,2.3,None,"POLYGON ((74.84492 28.873, 74.84492 28.87303, ..."
2,6.354292,4,google,0.7123,POINT (423693.943 3056529.04),74.226555,27.630664,74.22655544425939:27.630663527650903,0.000126,1.722222,...,4.21,10.1,District hospital Nagaur - dis_h,80.56,966.7,Parawa - sub_cen,2.90,34.8,None,"POLYGON ((74.22657 27.63066, 74.22656 27.63067..."
3,6.957619,4,google,0.7604,POINT (501374.223 3134297.669),75.014020,28.334861,75.0140202674914:28.334861398658834,0.000126,0.000000,...,1.85,4.6,D B Government Hospital Churu - dis_h,8.53,102.4,Boontia - sub_cen,1.85,22.2,None,"POLYGON ((75.01403 28.33486, 75.01403 28.33487..."
4,8.217702,4,google,0.7845,POINT (457645.79 3073203.448),74.570094,27.782683,74.57009354210085:27.782682537523247,0.000127,2.500000,...,0.31,0.5,"S K Hospital, Sikar - dis_h",69.99,839.8,Badawar - phc,0.31,3.8,None,"POLYGON ((74.5701 27.78267, 74.5701 27.7827, 7..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1481070,288.940417,4,google,0.9654,POINT (548170.125 3189797.089),75.493774,28.834925,75.49377437705633:28.83492532996736,0.124482,5.496721,...,0.51,0.8,Maharaja Aggarsain Medical College AGROHA - dis_h,53.42,641.0,Nyangal Chhoti - sub_cen,0.51,6.2,None,"POLYGON ((75.49442 28.83487, 75.4944 28.83505,..."
1481075,413.929305,16,microsoft,NaN,POINT (547761.161 3160531.774),75.488354,28.570791,75.48835436178372:28.570790796491007,0.128104,0.000000,...,0.87,1.3,B.D.K. Hospital Jhunjhunun - dis_h,74.42,893.1,Kalari - phc,0.87,10.4,None,"POLYGON ((75.48861 28.5707, 75.48865 28.57068,..."
1481083,289.408869,24,google,0.9092,POINT (493434.283 3119927.267),74.933096,28.205125,74.93309554461749:28.205124977994227,0.135970,8.185522,...,2.39,3.8,D B Government Hospital Churu - dis_h,13.48,161.8,Ratannagar - chc,2.39,28.6,None,"POLYGON ((74.93284 28.20498, 74.93285 28.20498..."
1481106,351.817061,31,microsoft,NaN,POINT (547943.937 3160388.159),75.490217,28.569488,75.49021717721772:28.56948776087451,0.158354,0.000000,...,0.70,1.1,B.D.K. Hospital Jhunjhunun - dis_h,74.26,891.1,Kalari - phc,0.70,8.4,None,"POLYGON ((75.48981 28.56982, 75.48979 28.56983..."


In [8]:
# Uses spatial join to assign each res building the ward name of the polygon it falls within.
# Buildings that don't fall inside any ward polygon are removed
#--------------------------------------------------------------------------------------------
joined = gpd.sjoin(churu_res_buildings, gdf_boundaries, how="left", predicate="within")
joined_wo_nan = joined[joined['Name'].notna()]
churu_res_buildings = joined_wo_nan

In [9]:
churu_res_buildings.columns

Index(['perimeter_in_meters', 'building_faces', 'bf_source', 'confidence_left',
       'geometry', 'longitude', 'latitude', 'id', 'area_in_meters',
       'height_mean', 'height_median', 'height_max', 'height', 'floors',
       'gfa_in_meters', 'urban_split', 'ghsl_smod', 'elevation',
       'building_density_50', 'building_density_100', 'building_density_250',
       'building_density_500', 'building_perimeter_in_meters_new',
       'perimeter_to_area_ratio', 'normalized_perimeter_to_area_ratio',
       'centroid', 'radius_m', 'num_vertices', 'classification_source',
       'osm_type', 'centroid_x', 'centroid_y', 'nearest_road_type_1',
       'distance_to_1', 'nearest_road_type_2', 'distance_to_2',
       'nearest_road_type_3', 'distance_to_3', 'nearest_road_type_4',
       'distance_to_4', 'road_density_for_4_fixed', 'road_density_for_5_fixed',
       'SQN', 'faces', 'prediction', 'confidence_settlement_clasification',
       'settlement_clasification', 'Tree_Cover_TC_air_distance_km

In [10]:
#file from MHT
#!pip install openpyxl
import openpyxl

population_churu = pd.read_excel(r"Churu Population.xlsx")
population_churu

,Ward Number,Total Population,Female,Male,Illiterate Population,Shape_Area (sq m),Area (sq_km),Density,Pop_2025
0,1,1873,899.04,973.96,505.71,2.305656e+06,2.305656,812.350066,2687.755
1,2,1830,878.40,951.60,494.10,2.590953e+05,0.259095,7063.038622,2626.050
2,3,1952,936.96,1015.04,527.04,1.642186e+05,0.164219,11886.591651,2801.120
3,4,1816,871.68,944.32,490.32,1.740005e+06,1.740005,1043.675241,2605.960
4,5,1852,888.96,963.04,500.04,7.908319e+04,0.079083,23418.376475,2657.620
...,...,...,...,...,...,...,...,...,...
56,57,2233,1071.84,1161.16,602.91,1.331904e+05,0.133190,16765.466442,3204.355
57,58,2334,1120.32,1213.68,630.18,1.226817e+05,0.122682,19024.842586,3349.290
58,59,1881,902.88,978.12,507.87,1.000343e+06,1.000343,1880.355270,2699.235
59,60,2042,980.16,1061.84,551.34,2.942562e+05,0.294256,6939.530709,2930.270


In [11]:
# Compute total ground floor area
#---------------------------------
total_gfa = churu_res_buildings['gfa_in_meters'].sum()
total_gfa

np.float64(3121252.2948100003)

In [12]:
#average gfa per inhabitant 
total_population = 171979.010
average_area_per_inhabitant = total_gfa / total_population
average_area_per_inhabitant

np.float64(18.14903048232456)

In [13]:
# Estimate inhabitants per res building for Churu and adds new column (inhabitants_whole_churu)
#-----------------------------------------------------------------------------------------------
churu_res_buildings["inhabitants_whole_churu"] = (churu_res_buildings["gfa_in_meters"] / total_gfa) * total_population
churu_res_buildings['Name'].astype(int)
churu_res_buildings['Name'].astype(str)

/usr/local/lib/python3.12/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


43         47
69          1
112        10
147        17
160        51
           ..
1480608    16
1480617    23
1480660     1
1480666    19
1480732    29
Name: Name, Length: 33577, dtype: object

In [15]:
# without decimal numbers
# This cell creates a whole number estimate of people per res building for each ward, while ensuring the sum per ward equals the value in Pop_2025 in the excel file
#--------------------------------------------------------------------------------------------------------------------------------------------------------------------

churu_res_buildings = churu_res_buildings.copy()
churu_res_buildings["inhabitants_with_integer_estimate"] = np.nan

# loop through each ward and filter res buildings that belong to that ward
for ward in range(1, 61):
    try:
        mask = churu_res_buildings["Name"].astype(int) == ward
    except:
        mask = churu_res_buildings["Name"].astype(str) == str(ward)
    
    df_ward = churu_res_buildings.loc[mask].copy()

    # Get the ward's population data from excel and skip wards with missing data
    try:
        pop_row = population_churu[population_churu["Ward Number"].astype(int) == ward]
    except:
        pop_row = population_churu[population_churu["Ward Number"].astype(str) == str(ward)]
    
    if df_ward.empty or pop_row.empty:
        continue
    
    total_pop = int(round(pop_row["Pop_2025"].values[0]))
    total_gfa = df_ward["gfa_in_meters"].sum() # total gfa per ward
    if total_gfa == 0:
        continue

    df_ward["ROcc"] = (df_ward["gfa_in_meters"] / total_gfa) * total_pop # raw continous population estimate
    df_ward["IOcc"] = np.floor(df_ward["ROcc"]).astype(int) # takes the integer part (rounds down) of each ROcc value
    df_ward["FOcc"] = df_ward["ROcc"] - df_ward["IOcc"] # Fractional remainder

    # Compute the number of people that still need to be added and allocate based on the deserve factor
    deficit = int(round(total_pop - df_ward["IOcc"].sum()))
    if deficit <= 0:
        churu_res_buildings.loc[mask, "inhabitants_with_integer_estimate"] = df_ward["IOcc"].values
        continue

    df_ward["DeserveFactor"] = np.where(df_ward["IOcc"] > 0, df_ward["FOcc"] / df_ward["IOcc"], 1.0)

    top_idx = df_ward["DeserveFactor"].nlargest(deficit).index
    df_ward.loc[top_idx, "IOcc"] += 1

    churu_res_buildings.loc[mask, "inhabitants_with_integer_estimate"] = df_ward["IOcc"].values


In [16]:
churu_res_buildings['inhabitants_with_integer_estimate']


43           1.0
69           1.0
112          1.0
147          1.0
160          1.0
           ...  
1480608     70.0
1480617    111.0
1480660     37.0
1480666     38.0
1480732     47.0
Name: inhabitants_with_integer_estimate, Length: 33577, dtype: float64

In [17]:
#without decimal numbers + informal settlements constant
# This cell generates a new column (inhabitants_with_integer_informal) that estimates per-building integer inhabitants, using the same logic as the prev cell, 
# but with an extra weight (informal_multiplier = 4) applied to buildings classified as informal.
#---------------------------------------------------------------------------------------------------------------------------------------------------------------
churu_res_buildings = churu_res_buildings.copy()
churu_res_buildings["inhabitants_with_integer_informal"] = np.nan

informal_multiplier = 4

for ward in range(1, 61):
    mask = churu_res_buildings["Name"].astype(str) == str(ward)
    df_ward = churu_res_buildings.loc[mask].copy()
    pop_row = population_churu[population_churu["Ward Number"] == ward]
    
    if df_ward.empty or pop_row.empty:
        continue
    
    total_pop = int(round(pop_row["Pop_2025"].values[0]))
    total_gfa = df_ward["gfa_in_meters"].sum()
    if total_gfa == 0:
        continue
    
    df_ward["weight"] = np.where(df_ward["settlement_clasification"].str.lower() == "informal", informal_multiplier, 1)
    df_ward["weighted_gfa"] = df_ward["gfa_in_meters"] * df_ward["weight"]
    total_weighted_gfa = df_ward["weighted_gfa"].sum()
    

    df_ward["ROcc"] = (df_ward["weighted_gfa"] / total_weighted_gfa) * total_pop
    

    df_ward["IOcc"] = np.floor(df_ward["ROcc"]).astype(int)
    df_ward["FOcc"] = df_ward["ROcc"] - df_ward["IOcc"]
    
    deficit = int(round(total_pop - df_ward["IOcc"].sum()))
    if deficit <= 0:
        churu_res_buildings.loc[mask, "inhabitants_with_integer_informal"] = df_ward["IOcc"].values
        continue

    df_ward["DeserveFactor"] = np.where(df_ward["IOcc"] > 0, df_ward["FOcc"] / df_ward["IOcc"], 1)
    
    top_idx = df_ward["DeserveFactor"].nlargest(deficit).index
    df_ward.loc[top_idx, "IOcc"] += 1
    

    churu_res_buildings.loc[mask, "inhabitants_with_integer_informal"] = df_ward["IOcc"].values


In [18]:
# Save only res buildings inside the wards with population estimates 
#--------------------------------------------------------------------
churu_res_buildings['geometry'] = churu_res_buildings['geometry2']
res_with_pop = churu_res_buildings.copy()
print(res_with_pop.geometry.total_bounds)


res_with_pop.set_crs("EPSG:4326", inplace=True, allow_override=True)

# # Save as Parquet
output_path = "churu_res_60_wards_with_population.parquet"
res_with_pop.to_parquet(output_path, index=False, engine="pyarrow")

print(res_with_pop.crs)
print(res_with_pop.geometry.geom_type.value_counts())

res_with_pop.head()


[74.92465697 28.26977178 75.00352777 28.3232721 ]
EPSG:4326
Polygon    33577
Name: count, dtype: int64


,perimeter_in_meters,building_faces,bf_source,confidence_left,geometry,longitude,latitude,id,area_in_meters,height_mean,...,Name,bvnvb,Shape_Leng,Shape_Area,Ward_sqkm,Percentage,Per_inter,inhabitants_whole_churu,inhabitants_with_integer_estimate,inhabitants_with_integer_informal
43,6.881646,4,google,0.7285,"POLYGON ((74.97421 28.30772, 74.97421 28.30774...",74.974198,28.307729,74.9741978462364:28.307729275278096,0.000136,2.166667,...,47,47.0,3027.168581,2.010054e+05,0.201,NaN,0.995,0.149795,1.0,1.0
69,7.007497,4,google,0.7928,"POLYGON ((74.9514 28.3073, 74.9514 28.30732, 7...",74.951391,28.307306,74.9513908477859:28.30730566614702,0.000141,0.000000,...,1,1.0,8789.786444,2.305656e+06,2.307,4.725,8.626,0.155496,1.0,1.0
112,6.909840,4,google,0.7027,"POLYGON ((74.96247 28.29955, 74.96247 28.29957...",74.962459,28.299556,74.96245903512192:28.2995564344419,0.000149,4.222222,...,10,10.0,1470.395664,6.481162e+04,0.065,NaN,98.462,0.163704,1.0,1.0
147,7.235648,4,google,0.7013,"POLYGON ((74.96398 28.29727, 74.96398 28.29729...",74.963972,28.297278,74.96397245063798:28.297277583031956,0.000154,6.900000,...,17,17.0,1810.881476,1.258111e+05,0.126,NaN,2.381,0.339073,1.0,1.0
160,7.054727,4,google,0.7199,"POLYGON ((74.97085 28.31411, 74.97084 28.31412...",74.970836,28.314113,74.97083611088743:28.314113464876772,0.000156,2.166667,...,51,51.0,1511.571591,5.941558e+04,0.059,NaN,1.695,0.171375,1.0,1.0


In [19]:
# #adding back geometry + nonres buildings
# churu_res_buildings['geometry'] = churu_res_buildings['geometry2'] 
# merged = pd.concat([churu_res_buildings, churu_other_buildings], ignore_index=True)

In [20]:
# merged_gdf = gpd.GeoDataFrame(merged, geometry='geometry')
# merged_gdf = merged_gdf.to_crs(epsg=4326)
# merged_gdf.to_parquet("churu_60_wards_with_population.parquet")

In [21]:
# merge and save the complete dataset including res buildings inside wards, other res buildings outside the ward boundaries, non-res buildings
#-----------------------------------------------------------------------------------------------------------------------------------------------

# Other residential buildings outside wards (those dropped earlier) 
churu_other_res = joined[joined["Name"].isna()].copy()
churu_other_res["geometry"] = churu_other_res["geometry2"] # restore polygons

# Ensure CRS is consistent
for gdf in [churu_res_buildings, churu_other_buildings, churu_other_res]:
    gdf.set_crs("EPSG:4326", inplace=True, allow_override=True)

# Merge all subsets 
merged = pd.concat([churu_res_buildings, churu_other_buildings, churu_other_res], ignore_index=True)

# Verify count and save
print("Total merged buildings:", len(merged))

output = "churu_buildings_with_population.parquet"
merged.to_parquet(output, index=False, engine="pyarrow")

print(f"Saved all buildings to: {output}")


/usr/local/lib/python3.12/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Total merged buildings: 1481134
Saved all buildings to: churu_buildings_with_population.parquet
